In [1]:
# Fold 0 grouped-model timing run: baseline hyperparameters
from pathlib import Path
import copy
import random
import time

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

repo = Path.cwd()
while not (repo / "data/ready_full/fold0").is_dir():
    if repo == repo.parent:
        raise FileNotFoundError("Could not find data/ready_full/fold0")
    repo = repo.parent

import sys
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from model.grouped_cnn_gru_attention import FullSensorGroupedCNNGRUAttention
from utils.evaluation import competition_score

FOLD = 0
SEED = 42
BATCH_SIZE = 16
MAX_EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
MIN_LEARNING_RATE = 1e-5
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 0  # Simple and reliable for Windows notebooks.

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; this timing run is intended for your GPU.")
device = torch.device("cuda")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"Fold {FOLD}, seed {SEED}, device: {torch.cuda.get_device_name(0)}")


Fold 0, seed 42, device: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
# Load only the selected fold. Memory mapping avoids copying the ~2 GB training array into RAM.
class GestureDataset(Dataset):
    def __init__(self, fold_dir, split):
        self.x = np.load(fold_dir / f"X_{split}.npy", mmap_mode="r")
        self.mask = np.load(fold_dir / f"mask_{split}.npy", mmap_mode="r")
        self.y = np.load(fold_dir / f"y_gesture_{split}.npy", mmap_mode="r")
        if not (len(self.x) == len(self.mask) == len(self.y)):
            raise ValueError(f"Mismatched {split} array lengths")

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        # Make writable per-sample copies before converting read-only memmap slices to tensors.
        x = torch.from_numpy(np.array(self.x[index], dtype=np.float32, copy=True))
        mask = torch.from_numpy(np.array(self.mask[index], copy=True))
        y = torch.tensor(int(self.y[index]), dtype=torch.long)
        return x, mask, y

fold_dir = repo / "data/ready_full" / f"fold{FOLD}"
train_dataset = GestureDataset(fold_dir, "train")
val_dataset = GestureDataset(fold_dir, "val")
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Train: {len(train_dataset)} sequences; validation: {len(val_dataset)} sequences")


Train: 6427 sequences; validation: 1627 sequences


In [3]:
# Grouped CNN-GRU-attention model with the same 18-class output as the baseline.
model = FullSensorGroupedCNNGRUAttention(num_classes=18).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE,
)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Trainable parameters: 1,591,728


In [4]:
from utils.train import train_one_epoch, validate

# Run ONE fold. Stop when the competition score has not improved for PATIENCE epochs.
history = []
best_score = float("-inf")
best_epoch = 0
best_metrics = None
best_weights = None
epochs_without_improvement = 0

torch.cuda.synchronize()
run_start = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, GRAD_CLIP_NORM)
    val_loss, metrics, _, _ = validate(model, val_loader, criterion, device)
    scheduler.step()
    torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - epoch_start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        **metrics,
        "seconds": epoch_seconds,
    })
    print(
        f"Epoch {epoch:02d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} "
        f"| binary F1 {metrics['binary_f1']:.4f} | macro F1 {metrics['macro_f1']:.4f} "
        f"| score {metrics['score']:.4f} | {epoch_seconds:.1f}s"
    )

    if metrics["score"] > best_score:
        best_score = metrics["score"]
        best_epoch = epoch
        best_metrics = metrics.copy()
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping after {PATIENCE} epochs without improvement.")
            break

fold_seconds = time.perf_counter() - run_start
model.load_state_dict(best_weights)
print(f"Best epoch: {best_epoch}")
print(f"Best binary F1: {best_metrics['binary_f1']:.4f}")
print(f"Best macro F1: {best_metrics['macro_f1']:.4f}")
print(f"Best validation score: {best_score:.4f}")
print(f"Fold {FOLD} training + validation time: {fold_seconds / 60:.1f} minutes")
print(f"Rough five-fold estimate: {5 * fold_seconds / 60:.1f} minutes")
print(f"Difference from existing fold-0 ToF score (0.8273): {best_score - 0.8273:+.4f}")


Epoch 01 | train loss 1.4771 | val loss 1.0968 | binary F1 0.9768 | macro F1 0.5245 | score 0.7506 | 63.4s
Epoch 02 | train loss 0.9222 | val loss 0.8914 | binary F1 0.9717 | macro F1 0.6029 | score 0.7873 | 47.9s
Epoch 03 | train loss 0.7720 | val loss 0.8212 | binary F1 0.9727 | macro F1 0.6426 | score 0.8077 | 76.9s
Epoch 04 | train loss 0.6913 | val loss 0.7944 | binary F1 0.9892 | macro F1 0.6605 | score 0.8248 | 44.3s
Epoch 05 | train loss 0.6209 | val loss 0.7721 | binary F1 0.9912 | macro F1 0.6629 | score 0.8270 | 42.9s
Epoch 06 | train loss 0.5534 | val loss 0.8187 | binary F1 0.9867 | macro F1 0.6479 | score 0.8173 | 42.8s
Epoch 07 | train loss 0.5233 | val loss 0.8070 | binary F1 0.9808 | macro F1 0.6786 | score 0.8297 | 42.8s
Epoch 08 | train loss 0.4816 | val loss 0.8080 | binary F1 0.9857 | macro F1 0.6876 | score 0.8366 | 54.9s
Epoch 09 | train loss 0.4399 | val loss 0.8343 | binary F1 0.9852 | macro F1 0.6902 | score 0.8377 | 43.4s
Epoch 10 | train loss 0.4039 | val lo